In [1]:
import sys
sys.path.append(".")

from utils import load_prices, compute_returns, build_network
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print("Ready")

Ready


In [2]:
prices = load_prices(start="2019-01-01", end="2024-12-31")
returns = compute_returns(prices)

print(f"Prices shape: {prices.shape}")
print(f"Returns shape: {returns.shape}")

[*********************100%***********************]  55 of 55 completed


Prices shape: (2191, 55)
Returns shape: (2190, 55)


The signal logic :

On day 3 we learned that betweenness centrality measures how often an assets sits on the critical path between other assets. High betweenness = systemic bridge = it it crashes, contagion spreads everywhere.

The trading hypothesis is :
Stocks with low betweenness are safer to hold because they are peripheral to the network - a crash elsewhere is less likely to drag them down. Stocks with high betweenness are dangerous to hold because they are contagion bridges.

So every weel we will :
-   Rank all stocks by betweenness centrality using the past 60 days
-   Go long the 10 stocks with the lowest betweenness - peripheral, isolated, low systemic risk
-   Go short the 10 stocks with the highest betweenness - central bridges, high systemic risk
-   Hold for one week then rebalance

It is a classic long/short equity strategy. The bet is that peripheral stocks outperform central stocks on a risk-adjusted basis because they carry less hidden systemic risk.

Computing weekly betweenness signals

In [4]:
window = 60
rebalance_freq = 5 #trading days in a week
top_n = 10

signal_dates = []
long_portfolios = []
short_portfolios = []

for i in range(window, len(returns) - rebalance_freq, rebalance_freq):
    window_returns = returns.iloc[i - window:i]
    G, _ = build_network(window_returns, threshold=0.4)

    betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
    betweenness_series = pd.Series(betweenness).sort_values()

    valid_stocks = [t for t in betweenness_series.index 
                    if t not in ["BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD"]]
    
    long_stocks = valid_stocks[:top_n]
    short_stocks = valid_stocks[-top_n:]

    signal_dates.append(returns.index[i])
    long_portfolios.append(long_stocks)
    short_portfolios.append(short_stocks)

print(f"Generated {len(signal_dates)} weekly signals")
print(f"\nFirst signal date: {signal_dates[0].date()}")
print(f"Long: {long_portfolios[0]}")
print(f"Short: {short_portfolios[0]}")


Generated 425 weekly signals

First signal date: 2019-03-03
Long: ['BAC', 'MCD', 'IBM', 'KO', 'GE', 'TSLA', 'WMT', 'GS', 'AMGN', 'QCOM']
Short: ['HON', 'ACN', 'CAT', 'ORCL', 'JPM', 'PEP', 'BA', 'UPS', 'COST', 'AAPL']


Let's me explain what we've done here : 
Every 5 trading days you look back at the past 60 days, build a network, rank stocks by betweenness centrality, and decide who to buy and who to sell short for the coming week. This repeats 425 times across 6 years.

Basically speaking, you take your first 5 days and to decide between stocks that go long/short for the week (those 5 days), you use the last 60 days data to compute the betweenness centrality, rank the stocks according to that measure and allocate the 10 first and 10 last to the short/long portfolios. And you move 5 days (next week), and then take the last 60 days preceding the first day of this new week and do the same computation. You do this to the end of the whole 6 years period.

Computing portofolio returns

In [5]:
strategy_returns = []
long_returns = []
short_returns = []

for i, date in enumerate(signal_dates):
    date_idx = returns.index.get_loc(date)
    
    if date_idx + rebalance_freq >= len(returns):
        break
    
    next_week = returns.iloc[date_idx:date_idx + rebalance_freq]
    
    long_stocks  = [s for s in long_portfolios[i]  if s in next_week.columns]
    short_stocks = [s for s in short_portfolios[i] if s in next_week.columns]
    
    long_ret  = next_week[long_stocks].mean(axis=1).sum()
    short_ret = next_week[short_stocks].mean(axis=1).sum()
    
    strategy_ret = long_ret - short_ret
    
    strategy_returns.append(strategy_ret)
    long_returns.append(long_ret)
    short_returns.append(short_ret)

results = pd.DataFrame({
    "strategy": strategy_returns,
    "long_leg": long_returns,
    "short_leg": short_returns
}, index=signal_dates[:len(strategy_returns)])

print(f"Computed {len(results)} weekly returns")
print(f"\nMean weekly strategy return: {results['strategy'].mean():.4f}")
print(f"Std weekly strategy return:  {results['strategy'].std():.4f}")

Computed 425 weekly returns

Mean weekly strategy return: 0.0004
Std weekly strategy return:  0.0119


In [ ]:
# results.to_csv("data/results.csv")
# print("results saved to data/results.csv")

Quick explanation of what we are doing here : 

We have our weekly decisions (portfolios) over the 6 years. Now the question to ask is "If we have opted for such strategy for real, what return would we have made each week ?"

We take the first week (first signal week), compute the avg across stocks (long and short) for every day and then sum over 5 days for the weekly return (because it is log, we just sum). 

Now about the strategy_ret:

strategy_ret = long_ret - short_ret

When you short a stock, you profit when it goes down. So a positive short actually means you lost money on the short leg. Substracting shor_ret from long_ret gives you the true P&L of the combined position. If long goes up and short goes down it means strategy_ret is positive. If both go up equally it means strategy_ret is near zero (hedged). If short goes up more than long it means strategy_ret is negative.

We also computed the mean weekly return (over the whole period). It is 0.0004 meaning the strategy made on average 0.04% per week (small but we need to see it compounded over time and compared to the benchmark before drawing any conclusions)
